In [ ]:
import sqlite3
import pandas as pd

In [2]:
# File paths
CSV_FILE = "../../data/processed/processed_data.csv"
DB_FILE = "../../data/database.db"
TABLE_NAME = "records"

In [7]:
# Connect to SQLite
conn = sqlite3.connect(DB_FILE)

In [4]:
# Drop table if already exists
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
conn.commit()

In [5]:
# Read CSV in chunks to handle large files
chunksize = 100_000
for i, chunk in enumerate(pd.read_csv(CSV_FILE, chunksize=chunksize)):
    print(f"Processing chunk {i+1}...")
    chunk.to_sql(TABLE_NAME, conn, if_exists="append", index=False)

# Create index for fast lookup on Customer
cursor = conn.cursor()
cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_customer ON {TABLE_NAME}(Customer)")
conn.commit()

Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...


In [6]:
# Close connection
conn.close()
print("Database created successfully.")

Database created successfully.


In [8]:
def get_last_entry(customer_name):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute(f"""
        SELECT *
        FROM {TABLE_NAME}
        WHERE Customer = ?
        ORDER BY rowid DESC
        LIMIT 1
    """, (customer_name,))

    row = cursor.fetchone()
    conn.close()
    return row

In [11]:
# read latest entry for customer
print(get_last_entry('C1166355595'))

(179, 'C1166355595', 'M348934600', 48.47, 0, 178.0, 1.0, 177, 6428.629999999999, 36.319943502824856, 84.36885925463618, 1.3345285076291529, 0.847809946803922, 0.1440111506131035, 67, 149, 178.0, 1.0, 205333, 5537053.38, 26.96621283476109, 17.52629536169834, 1.7974344524018298, 1.0287027301526124, 1.226944241293166, 0, 0.0, 179.0, 0.0, 22512281.539999984, 37.89512570951471, 1.2790563190513484, 0.8237614621351912, 111.45234547849188, 0.0948824741649428, 26.9, 1.8018587360594795, 1.030283031247585, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0)
